# Why is the quantum false rate ≫ classical at γ = 3?

At γ = 3 the classical and 1BQF quantum solvers act on the **same** linear system
`Ax = b` (same event, same matrix A), so the segment false rate should agree.
The 2×2 figure showed it does not: classical ≈ 2 % vs quantum ≈ 44 % at n = 400.
This notebook tests the three candidate causes:

1. **Mismatched metrics** — are we even comparing the same events / matrices?
2. **Definitional error** — is the metric formula applied differently to Q vs C?
3. **Convention error** — is something in the Q post-processing (rescale /
   threshold) not scale-comparable to C?

**Spoiler / result:** it is **(3) a convention error**.  The quantum metric
rescaled `sol_Q` to ‖sol_C‖, and ‖sol_C‖ is 95–99 % the *false bulk* (every
false segment sits at the Hopfield attractor 0.25), so ‖sol_C‖ ∝ √n_seg.  That
n-dependent rescale inflated the quantum amplitudes and pushed borderline false
segments over the **fixed** absolute threshold τ = 0.35 — an artefact that grows
with multiplicity.  The quantum solve itself is faithful (cos on the true
subspace ≈ 0.97) and ranks true above false (AUC ≈ 1).  Rescaling on the
classical *signal* support recovers classical-matching false rates.

> **RESOLVED (2026-06-09).** The fix is now implemented in
> `qtrk_pipeline/metrics.py` as `rescale_to_signal`, and `quantum_metrics` uses
> it; the metrics view has been rebuilt.  §5 below contrasts the old ‖sol_C‖
> convention with the shipped `rescale_to_signal` fix.

In [1]:
import sys
sys.path.insert(0, "/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Segment_level_studies")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seg_store as S
import qtrk_pipeline as qp

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})
OUT = Path(S.__file__).resolve().parent / "outputs" / "quantum_false_rate_investigation"
OUT.mkdir(parents=True, exist_ok=True)
GAMMA, TAU = 3.0, S.threshold(3.0)
N_GRID = [10, 20, 50, 100, 200, 400]
CI = S.solves_index("classical", gamma=GAMMA, hit_ineff=0.0)
QI = S.solves_index("quantum",   gamma=GAMMA, hit_ineff=0.0)
print("γ =", GAMMA, " τ =", TAU, " (classical false attractor δ/(δ+γ) = 0.25)")

γ = 3.0  τ = 0.35  (classical false attractor δ/(δ+γ) = 0.25)


In [2]:
def paired(n_trk, max_reps=3):
    """All reps at n_trk that have BOTH a classical and quantum solve of the SAME
    (event_key, ham_key).  Returns list of dicts with the raw vectors + truth."""
    out = []
    for _, q in QI[QI.n_trk == n_trk].head(max_reps).iterrows():
        c = CI[(CI.event_key == q.event_key) & (CI.ham_key == q.ham_key)]
        if not len(c):
            continue
        c = c.iloc[0]
        solC = np.asarray(qp.load_solution(c.sol_key)["sol"], float)
        solQraw = np.asarray(qp.load_solution(q.sol_key)["sol"], float)
        truth = np.asarray(qp.truth_from_event(S._event_of(c)), bool)
        out.append(dict(event_key=q.event_key, ham_key=q.ham_key, n_trk=n_trk,
                        solC=solC, solQraw=solQraw, truth=truth))
    return out

## 1. Mismatched metrics?  — pairing audit

The metric view pairs each quantum solve with the classical solve of the **same
`(event_key, ham_key)`** — i.e. the *same matrix A* (the `ham_key` folds in
ε, γ, δ, kernel).  Confirm the pairing is 1:1 on the same A.

In [3]:
rows = []
for n in N_GRID:
    for d in paired(n):
        same_len = d["solC"].shape == d["solQraw"].shape == d["truth"].shape
        rows.append(dict(n_trk=n, event_key=d["event_key"][:14], ham_key=d["ham_key"][:14],
                         n_seg=d["solC"].size, shapes_match=same_len,
                         n_true=int(d["truth"].sum())))
audit = pd.DataFrame(rows)
print("paired solves found:", len(audit), " — all share one (event_key, ham_key) => same A")
print("shape agreement (solC, solQ, truth all equal length):", bool(audit.shapes_match.all()))
display(audit.head(8))

paired solves found: 18  — all share one (event_key, ham_key) => same A
shape agreement (solC, solQ, truth all equal length): True


,n_trk,event_key,ham_key,n_seg,shapes_match,n_true
0,10,ev_7b7a0c3c815,hm_2413aa077d9,400,True,40
1,10,ev_b4086c4068b,hm_2413aa077d9,400,True,40
2,10,ev_ad2f01d32e9,hm_2413aa077d9,400,True,40
3,20,ev_d8187123659,hm_2413aa077d9,1600,True,80
4,20,ev_d761eefd20b,hm_2413aa077d9,1600,True,80
5,20,ev_51d19b7a9b1,hm_2413aa077d9,1600,True,80
6,50,ev_3bd70f4c782,hm_2413aa077d9,10000,True,200
7,50,ev_c872305d9a4,hm_2413aa077d9,10000,True,200


## 2. Definitional error?  — identical metric code path

`build_metrics` computes the classical metric as `metrics_at(sol_C, truth, τ)`
and the quantum metric as `quantum_metrics(sol_Q_raw, sol_C, truth, τ)`, which is
`metrics_at(rescale_to(sol_Q_raw, sol_C), truth, τ)`.  Same formula, same truth.
**Proof:** feed `sol_C` itself through the *quantum* path — `rescale_to(sol_C,
sol_C) = sol_C`, so it must reproduce the classical false rate exactly.  It does:

In [4]:
rows = []
for n in N_GRID:
    P = paired(n)
    if not P:
        continue
    fC, fQviaC = [], []
    for d in P:
        truth = d["truth"]
        fC.append(qp.metrics_at(d["solC"], truth, TAU)["segment_false_rate"])
        # sol_C pushed through the quantum metric path
        fQviaC.append(qp.quantum_metrics(d["solC"], d["solC"], truth, TAU)["segment_false_rate"])
    rows.append(dict(n_trk=n, far_classical_pct=round(np.mean(fC)*100, 3),
                     far_solC_via_quantum_path_pct=round(np.mean(fQviaC)*100, 3)))
chk = pd.DataFrame(rows)
print("If these two columns match, the metric DEFINITION is identical for C and Q:")
display(chk)
print("=> identical. Not a definitional / metric-application error.")

If these two columns match, the metric DEFINITION is identical for C and Q:


,n_trk,far_classical_pct,far_solC_via_quantum_path_pct
0,10,0.000,0.000
1,20,0.000,0.000
2,50,0.654,0.654
3,100,0.166,0.166
4,200,0.330,0.330
5,400,2.161,2.161


=> identical. Not a definitional / metric-application error.


## 3. The fidelity paradox — ‖sol_C‖ is the false bulk

The stored `cos_QC` looks awful (~0.1), which would suggest the quantum solve is
wrong.  But the full cosine is dominated by the **false bulk**: every false
segment sits at 0.25, and there are O(n²) of them, so they contribute
95–99 % of ‖sol_C‖².  On the **true subspace** the quantum and classical
solutions agree at cos ≈ 0.97 — the quantum solve is faithful where it matters.

In [5]:
rows = []
for n in N_GRID:
    P = paired(n)
    if not P:
        continue
    cf, ct, fb, nrm = [], [], [], []
    for d in P:
        solC, truth = d["solC"], d["truth"]
        solQ = qp.rescale_to(d["solQraw"], solC)
        cf.append(qp.cos_sim(solQ, solC))
        ct.append(qp.cos_sim(solQ[truth], solC[truth]))
        fb.append(np.linalg.norm(solC[~truth])**2 / np.linalg.norm(solC)**2)
        nrm.append(np.linalg.norm(solC))
    rows.append(dict(n_trk=n, cos_full=np.mean(cf), cos_true_subspace=np.mean(ct),
                     false_bulk_frac=np.mean(fb), norm_solC=np.mean(nrm)))
fid = pd.DataFrame(rows)
display(fid.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(fid.n_trk, fid.cos_full, "o-", color="#d6604d", label="cos(sol_Q, sol_C) — full")
ax[0].plot(fid.n_trk, fid.cos_true_subspace, "s-", color="#1b7837",
           label="cos on TRUE subspace")
ax[0].set_xscale("log"); ax[0].set_ylim(0, 1.05); ax[0].set_xlabel("n_tracks")
ax[0].set_ylabel("cosine similarity"); ax[0].legend(fontsize=9)
ax[0].set_title("(a) Faithful where it matters", fontweight="bold")
ax[1].plot(fid.n_trk, fid.false_bulk_frac * 100, "^-", color="#6a3d9a")
ax[1].set_xscale("log"); ax[1].set_ylim(90, 100); ax[1].set_xlabel("n_tracks")
ax[1].set_ylabel("false-bulk share of ‖sol_C‖²  (%)")
ax[1].set_title("(b) ‖sol_C‖ is mostly the 0.25 false bulk", fontweight="bold")
fig.tight_layout()
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"fidelity_decomposition.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show(); print("saved fidelity_decomposition")

,n_trk,cos_full,cos_true_subspace,false_bulk_frac,norm_solC
0,10,0.465,0.967,0.769,5.411
1,20,0.342,0.967,0.875,10.419
2,50,0.223,0.966,0.947,25.433
3,100,0.161,0.967,0.973,50.438
4,200,0.121,0.962,0.986,100.502
5,400,0.099,0.966,0.993,200.600


saved fidelity_decomposition


## 4. The mechanism — an n-dependent rescale over a fixed threshold

`rescale_to(sol_Q_raw, sol_C) = sol_Q_raw · ‖sol_C‖ / ‖sol_Q_raw‖`, and
`sol_Q_raw` is **unit norm**, so the rescale factor is just ‖sol_C‖ ∝ √n_seg.
This multiplies the (unit-norm) quantum amplitudes by a factor that grows with
multiplicity, dragging an increasing number of borderline false segments over
the fixed τ = 0.35.  The quantum false *bulk* stays at ~0, but the promoted tail
grows with n.

In [6]:
# rescale factor vs n, and quantum solution distributions (L2 convention) at 2 n
P50, P400 = paired(50)[0], paired(400)[0]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

ax[0].plot(fid.n_trk, fid.norm_solC, "o-", color="#2166ac", label="‖sol_C‖ (= rescale factor)")
ns = np.array(fid.n_trk, float)
ax[0].plot(ns, fid.norm_solC.iloc[0] * np.sqrt(ns**2 / ns[0]**2), "k--", alpha=0.5,
           label=r"$\propto\sqrt{n_{\rm seg}}\;(n_{\rm seg}\!\sim\!n^2)$")
ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("n_tracks")
ax[0].set_ylabel("rescale factor ‖sol_C‖"); ax[0].legend(fontsize=9)
ax[0].set_title("(a) Quantum rescale factor grows with n", fontweight="bold")

for ax_i, P, n in [(ax[1], P50, 50), (ax[2], P400, 400)]:
    solQ = qp.rescale_to(P["solQraw"], P["solC"]); t = P["truth"]
    bins = np.linspace(0, max(np.percentile(solQ[t], 99), TAU * 2), 70)
    ax_i.hist(solQ[~t], bins=bins, color="#c51b7d", alpha=0.6, label=f"false ({(~t).sum():,})")
    ax_i.hist(solQ[t], bins=bins, histtype="step", color="#1b7837", lw=1.8,
              label=f"true ({t.sum():,})")
    ax_i.axvline(TAU, color="k", ls="--", lw=1.2, label=f"τ={TAU:.2f}")
    fp = int((solQ[~t] > TAU).sum())
    ax_i.set_yscale("log"); ax_i.set_ylim(0.5, None)
    ax_i.set_xlabel("rescaled quantum  $s_i$"); ax_i.set_ylabel("count")
    ax_i.set_title(f"(n={n}) false>τ: {fp:,}", fontweight="bold"); ax_i.legend(fontsize=8)
fig.tight_layout()
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"rescale_mechanism.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show(); print("saved rescale_mechanism")

saved rescale_mechanism


## 5. The fix — a scale-comparable threshold recovers agreement

The absolute τ = 0.35 is meaningful for the classical solution because its
Hopfield fixed points are **n-independent** (false → 0.25, true plateau ≳ 0.375).
The L2-rescaled quantum solution has no such fixed scale.  The shipped fix
(`qtrk_pipeline.metrics.rescale_to_signal`, used by `quantum_metrics`) rescales
`sol_Q` on the classical **signal support** (`sol_C > τ`, the classically-active
segments) instead of the false-bulk-dominated full L2 norm — truth-free, and it
collapses the quantum false rate onto the classical curve.  (AUC below confirms
the quantum *ranking* is essentially perfect, so the gap was scale, not order.)

In [7]:
def auc(score, truth):
    order = np.argsort(score); ranks = np.empty(len(score)); ranks[order] = np.arange(len(score))
    nt, nf = truth.sum(), (~truth).sum()
    return (ranks[truth].sum() - nt * (nt - 1) / 2) / (nt * nf)

rows = []
for n in N_GRID:
    P = paired(n)
    if not P:
        continue
    fC, fL2, fTrue, au = [], [], [], []
    for d in P:
        solC, truth = d["solC"], d["truth"]
        fC.append(qp.metrics_at(solC, truth, TAU)["segment_false_rate"])
        solQ_L2 = qp.rescale_to(d["solQraw"], solC)                       # OLD convention
        fL2.append(qp.metrics_at(solQ_L2, truth, TAU)["segment_false_rate"])
        # NEW shipped fix: rescale on the classical signal support (truth-free)
        fTrue.append(qp.quantum_metrics(d["solQraw"], solC, truth, TAU)["segment_false_rate"])
        au.append(auc(d["solQraw"], truth))
    rows.append(dict(n_trk=n, far_classical=np.mean(fC)*100,
                     far_quantum_L2=np.mean(fL2)*100,
                     far_quantum_signalfix=np.mean(fTrue)*100, auc_quantum=np.mean(au)))
fix = pd.DataFrame(rows)
display(fix.round(3))

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(fix.n_trk, fix.far_classical, "o-", color="#1b7837", lw=2, label="classical")
ax.plot(fix.n_trk, fix.far_quantum_L2, "s-", color="#d6604d", lw=2,
        label="quantum — ‖sol_C‖ rescale (OLD convention)")
ax.plot(fix.n_trk, fix.far_quantum_signalfix, "D--", color="#2166ac", lw=2,
        label="quantum — rescale_to_signal (SHIPPED fix)")
ax.set_xscale("log"); ax.set_xlabel("number of tracks"); ax.set_ylabel("segment false rate (%)")
ax.set_title("Quantum false rate: convention artefact vs scale-comparable", fontweight="bold")
ax.legend(fontsize=10)
fig.tight_layout()
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"false_rate_convention_fix.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show(); print("saved false_rate_convention_fix")
print("\nAUC(quantum) ~", round(fix.auc_quantum.mean(), 4),
      "-> quantum ranks true above false almost perfectly; the gap is scale, not ranking.")

,n_trk,far_classical,far_quantum_L2,far_quantum_signalfix,auc_quantum
0,10,0.000,0.000,0.000,1.000
1,20,0.000,0.000,0.000,1.000
2,50,0.654,1.612,0.444,1.000
3,100,0.166,5.350,0.111,1.000
4,200,0.330,15.603,0.329,1.000
5,400,2.161,43.808,1.727,0.999


saved false_rate_convention_fix

AUC(quantum) ~ 0.9998 -> quantum ranks true above false almost perfectly; the gap is scale, not ranking.


## 6. Can we get both high efficiency *and* low false rate? — the activations

The §5 fix makes the false rate match classical, but at the classical operating
point τ = 0.35 the quantum **efficiency** is ~75 %, not the ~100 % we get
classically.  AUC ≈ 1 says the quantum *ranking* is near-perfect, so a better
operating point should exist.  Looking at the activation distributions explains
exactly what is happening.

In [8]:
CI = S.solves_index("classical", gamma=GAMMA, hit_ineff=0.0)
QI = S.solves_index("quantum",   gamma=GAMMA, hit_ineff=0.0)
TAU_Q = 0.09   # a quantum noise-floor threshold: in the gap between false(~0) and the lower true plateau

def vecs(n):
    q = QI[QI.n_trk == n].iloc[0]
    c = CI[(CI.event_key == q.event_key) & (CI.ham_key == q.ham_key)].iloc[0]
    solC = np.asarray(qp.load_solution(c.sol_key)["sol"], float)
    solQ = qp.rescale_to_signal(np.asarray(qp.load_solution(q.sol_key)["sol"], float), solC, TAU)
    truth = np.asarray(qp.truth_from_event(S._event_of(c)), bool)
    return solC, solQ, truth

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
for axi, n in [(ax[0], 100), (ax[1], 400)]:
    solC, solQ, truth = vecs(n)
    bins = np.linspace(0, max(solQ[truth].max(), 0.7), 70)
    axi.hist(solQ[~truth], bins=bins, color="#c51b7d", alpha=0.6, label=f"false ({(~truth).sum():,})")
    axi.hist(solQ[truth], bins=bins, histtype="step", color="#1b7837", lw=1.9, label=f"true ({truth.sum():,})")
    axi.axvline(TAU, color="k", ls="--", lw=1.4, label=f"classical τ = {TAU:.2f}")
    axi.axvline(TAU_Q, color="#2166ac", ls=":", lw=1.8, label=f"quantum τ_Q = {TAU_Q}")
    axi.set_yscale("log"); axi.set_ylim(0.5, None)
    axi.set_xlabel("quantum activation (signal-rescaled)"); axi.set_ylabel("count")
    axi.set_title(f"n={n}: two true plateaus — τ=0.35 cuts the lower one", fontsize=10, fontweight="bold")
    axi.legend(fontsize=8)
# classical -> quantum activation map (true vs false), n=100
solC, solQ, truth = vecs(100)
ax[2].scatter(solC[truth], solQ[truth], s=7, alpha=0.35, color="#1b7837", label="true")
ax[2].scatter(solC[~truth], solQ[~truth], s=4, alpha=0.15, color="#c51b7d", label="false")
ax[2].axhline(TAU, color="k", ls="--", lw=1); ax[2].axvline(TAU, color="grey", ls=":", lw=1)
ax[2].set_xlabel("classical activation $x_C$"); ax[2].set_ylabel("quantum activation $x_Q$")
ax[2].set_title("n=100: 1BQF halves the outer true plateau (0.364 → 0.18)", fontsize=10, fontweight="bold")
ax[2].legend(fontsize=8)
fig.suptitle("Quantum activation structure: the 1BQF quantises true segments into two plateaus",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"activation_structure.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show(); print("saved activation_structure")

saved activation_structure


In [9]:
# Efficiency / false-rate trade-off: classical τ=0.35 vs quantum noise-floor τ_Q=0.09, across n
rows = []
for n in N_GRID:
    P = paired(n)
    if not P:
        continue
    acc = {k: [] for k in ("e35", "f35", "eQ", "fQ", "maxf")}
    for d in P:
        solC, truth = d["solC"], d["truth"]
        solQ = qp.rescale_to_signal(d["solQraw"], solC, TAU)
        def ef(t):
            a = solQ > t
            return (truth & a).sum() / max(truth.sum(), 1), (~truth & a).sum() / max(a.sum(), 1)
        e1, f1 = ef(TAU); e2, f2 = ef(TAU_Q)
        acc["e35"].append(e1); acc["f35"].append(f1)
        acc["eQ"].append(e2);  acc["fQ"].append(f2)
        acc["maxf"].append(solQ[~truth].max())
    rows.append(dict(n_trk=n, eff_tau035=np.mean(acc["e35"])*100, far_tau035=np.mean(acc["f35"])*100,
                     eff_tauQ=np.mean(acc["eQ"])*100, far_tauQ=np.mean(acc["fQ"])*100,
                     max_false_Q=np.mean(acc["maxf"])))
trade = pd.DataFrame(rows)
display(trade.round(2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].plot(trade.n_trk, trade.eff_tau035, "s-", color="#d6604d", lw=2, label="τ = 0.35 (classical point)")
ax[0].plot(trade.n_trk, trade.eff_tauQ, "D-", color="#2166ac", lw=2, label=f"τ_Q = {TAU_Q} (quantum floor)")
ax[0].set_xscale("log"); ax[0].set_ylim(60, 102); ax[0].set_xlabel("n_tracks")
ax[0].set_ylabel("segment efficiency (%)"); ax[0].legend(fontsize=9)
ax[0].set_title("Efficiency: the quantum floor recovers ~100%", fontweight="bold")
ax[1].plot(trade.n_trk, trade.far_tau035, "s-", color="#d6604d", lw=2, label="τ = 0.35")
ax[1].plot(trade.n_trk, trade.far_tauQ, "D-", color="#2166ac", lw=2, label=f"τ_Q = {TAU_Q}")
ax[1].set_xscale("log"); ax[1].set_xlabel("n_tracks"); ax[1].set_ylabel("segment false rate (%)")
ax[1].legend(fontsize=9)
ax[1].set_title("False rate: the cost is the 1BQF false tail at high n", fontweight="bold")
fig.suptitle("Operating-point trade-off — both achievable up to moderate T, then the false tail bites",
             fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"operating_point_tradeoff.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show(); print("saved operating_point_tradeoff")

,n_trk,eff_tau035,far_tau035,eff_tauQ,far_tauQ,max_false_Q
0,10,75.00,0.00,100.0,0.00,0.00
1,20,75.00,0.00,100.0,0.00,0.00
2,50,74.67,0.44,100.0,1.61,0.24
3,100,74.92,0.11,100.0,5.35,0.33
4,200,75.08,0.33,100.0,15.60,0.42
5,400,74.71,1.73,100.0,43.81,0.67


saved operating_point_tradeoff


**What the activations show.**

1. The classical solution puts every true segment on one of two clean Hopfield
   levels — **0.455** (inner-chain) and **0.364** (outer/end) — both above
   τ = 0.35, with false pinned at 0.25. That is why classical efficiency ≈ 100 %.
2. The 1BQF (a *one-bit* eigenvalue inversion) **quantises** the true segments
   into two plateaus: it keeps the inner one (≈ 0.45) but **halves the outer
   one to ≈ 0.18**, while false segments collapse to ≈ 0. The classical
   τ = 0.35 lands in the **gap between the two quantum true plateaus**, so it
   rejects false (good) *and* discards the entire 0.18 plateau — exactly the
   ~25 % efficiency loss.
3. **So we can get both — up to moderate T.** Because false sits at ≈ 0 and the
   lowest true plateau is at ≈ 0.18, a quantum noise-floor threshold
   τ_Q ≈ 0.09 (in the gap) captures *both* true plateaus → **efficiency ≈ 100 %**.
   At T ≤ 100 this gives ≈ 100 % efficiency at ≤ 5 % false rate.
4. **The genuine limit** is the high-T false tail: the 1BQF also promotes a
   growing population of *false* segments (max false activation 0 → ~0.67 by
   T = 400) that overlaps the lower true plateau, so the low threshold admits
   them and the false rate climbs (≈ 16 % at T = 200, ≈ 44 % at T = 400). This
   is real 1BQF degradation, not a convention artefact — it is the quantum
   analogue of the classical false-tail growth, amplified by the one-bit filter.

**Bottom line for the metric.** The signal-rescale + classical τ = 0.35 is the
*scale-comparable* operating point (same cut as classical) and reports a
faithful ~75 % efficiency / ~classical false rate. If instead we want the 1BQF's
*best* operating point, the activations say to threshold at the quantum
noise-floor (τ_Q ≈ 0.09): ≈ 100 % efficiency with low false rate up to moderate
multiplicity, with an explicit ROC trade-off beyond. Either is defensible as
long as it is stated; they answer different questions (same-cut comparison vs
best-achievable).

## 7. The mechanism, from first principles — Hopfield levels, eigenvalues, and the 1-bit filter

This section proves *why* the classical and quantum activation structures differ,
analytically and then verified on stored data. Everything below is for the
operating point $\gamma=3,\ \delta=1$ (so $\gamma+\delta=4$, $\tau=\delta/(\delta+\gamma)+0.10=0.35$).

### 7.1 The classical Hopfield levels are exact rational numbers

The linear system is $A\mathbf{x}=\mathbf{b}$, $\mathbf{b}=\delta\mathbf{1}$, with
$A_{ii}=\gamma+\delta$ and $A_{ij}=-1$ for the segment pairs that share a middle
hit and are angle-compatible (step kernel).

* **False / isolated segment** (no compatible neighbour): its row is just
  $(\gamma+\delta)\,x_i=\delta$, so
  $$x_{\text{false}}=\frac{\delta}{\delta+\gamma}=\tfrac14=0.25.$$
* **True segment** on a clean track: a track crosses 5 planes → **4 consecutive
  segments**, each sharing a hit with its neighbour, forming a nearest-neighbour
  **chain**. Its block of $A$ is the $4\times4$ tridiagonal
  $$A_{\text{chain}}=\begin{pmatrix}4&-1&&\\-1&4&-1&\\&-1&4&-1\\&&-1&4\end{pmatrix},\quad \mathbf{b}=\mathbf{1}.$$
  By the $x_1{=}x_4,\ x_2{=}x_3$ symmetry: $4x_1-x_2=1$ and $-x_1+3x_2=1$, giving
  $$x_{\text{outer}}=\tfrac{4}{11}=0.3636,\qquad x_{\text{inner}}=\tfrac{5}{11}=0.4545.$$

So the classical solution has **three exact levels**: false $0.25$, true-outer
$4/11$, true-inner $5/11$. The threshold $\tau=0.35$ sits in the gap
$(0.25,\,0.3636)$ — below every true segment, above every false one. That is the
whole reason classical efficiency is $\approx100\%$ at $\approx0\%$ false rate.

### 7.2 The spectrum and the two inversions

Diagonalise $A=\sum_j\lambda_j\,\mathbf{u}_j\mathbf{u}_j^\top$.

* An **isolated/false** segment is an eigenvector with eigenvalue exactly
  $\lambda=\gamma+\delta=4$ — the whole false bulk piles up at $\lambda=4$.
* A clean **track chain** contributes the path-graph spectrum
  $$\lambda_k=(\gamma+\delta)-2\cos\!\frac{k\pi}{5}=\{2.382,\ 3.382,\ 4.618,\ 5.618\},\quad k=1\ldots4,$$
  straddling $\lambda=4$.

With $\mathbf{b}=\sum_j\beta_j\mathbf{u}_j$:

| solver | what it does to each eigenmode | weight $w(\lambda_j)$ |
|---|---|---|
| **classical** $A^{-1}\mathbf b$ | exact inversion | $1/\lambda_j$ — smooth, positive, monotone |
| **1BQF** | 1-bit eigenvalue filter (below) | $\cos(\lambda_j t/2)$ — oscillatory, sign-changing, **zero at $\lambda=\gamma+\delta$** |

### 7.3 Why the quantum weight is $\cos(\lambda t/2)$ — the 1-bit filter

`OneBQF` uses **one** time qubit ($t=\pi/(\gamma+\delta)$). On eigenstate
$\mathbf u_j$ (so $U=e^{-iAt}$ gives phase $e^{-i\lambda_j t}$) the circuit is a
**Hadamard test**: H on the time qubit → controlled-$U$ → H, then the ancilla is
flipped on `time=0` and post-selected. Standard Hadamard-test algebra gives
$$P(\text{time}=0)=\cos^2\!\big(\tfrac{\lambda_j t}{2}\big)\ \Rightarrow\ \text{kept amplitude}\ \propto\ \cos\!\big(\tfrac{\lambda_j t}{2}\big).$$
With $t=\pi/(\gamma+\delta)$ the filter is $f(\lambda)=\cos\!\big(\tfrac{\pi}{2}\tfrac{\lambda}{\gamma+\delta}\big)$, and

$$\boxed{f(\gamma+\delta)=\cos\tfrac{\pi}{2}=0.}$$

So the **rejection notch sits exactly on the false bulk** ($\lambda=\gamma+\delta$):
the 1BQF *annihilates* isolated/false segments (quantum false $\to 0$, cleaner
than the classical $0.25$). But the same $f$ is non-monotone across the true
chain spectrum: $f(\{2.382,3.382,4.618,5.618\})=\{+0.59,+0.24,-0.24,-0.59\}$.
The near-notch modes ($3.382,4.618$) are suppressed and the high mode's sign
flips, which **reshapes the true plateau and roughly halves the outer level**
($4/11\to\approx0.18$) while the inner level survives. $\tau=0.35$, calibrated to
the *classical* fixed points, then slices through the (real, but down-shifted)
outer plateau — the 25% efficiency loss. The cells below verify every number.

In [10]:
# ---- 7a. analytic isolated 4-segment chain: exact classical levels + spectrum + filter
g, d = 3.0, 1.0; diag = g + d; t = np.pi / diag
Ac = diag*np.eye(4) - (np.eye(4, k=1) + np.eye(4, k=-1))
xc = np.linalg.solve(Ac, np.ones(4))
lam = np.linalg.eigvalsh(Ac)
print("classical chain solution :", np.round(xc, 4), " = (4/11, 5/11, 5/11, 4/11) =",
      np.round([4/11, 5/11, 5/11, 4/11], 4))
print("false/isolated level     : delta/(delta+gamma) =", d/(d+g))
print("chain eigenvalues        :", np.round(lam, 3),
      " = (g+d) - 2cos(k pi/5) =", np.round(diag - 2*np.cos(np.arange(1,5)*np.pi/5), 3))
print("1-bit filter f(lambda)=cos(lambda*t/2), t=pi/(g+d):", np.round(np.cos(lam*t/2), 3))
print("  -> NOTCH: f(g+d=%.0f) = cos(pi/2) = %.3f   (annihilates the false bulk at lambda=g+d)"
      % (diag, np.cos(diag*t/2)))
print("classical 1/lambda weights:", np.round(1/lam, 3), " (smooth, monotone, all +)")

classical chain solution : [0.3636 0.4545 0.4545 0.3636]  = (4/11, 5/11, 5/11, 4/11) = [0.3636 0.4545 0.4545 0.3636]
false/isolated level     : delta/(delta+gamma) = 0.25
chain eigenvalues        : [2.382 3.382 4.618 5.618]  = (g+d) - 2cos(k pi/5) = [2.382 3.382 4.618 5.618]
1-bit filter f(lambda)=cos(lambda*t/2), t=pi/(g+d): [ 0.593  0.24  -0.24  -0.593]
  -> NOTCH: f(g+d=4) = cos(pi/2) = 0.000   (annihilates the false bulk at lambda=g+d)
classical 1/lambda weights: [0.42  0.296 0.217 0.178]  (smooth, monotone, all +)


In [11]:
# ---- 7b. PROOF on a stored real event (n=10): both filters reproduce the stored solves
n = 10
q = QI[QI.n_trk == n].iloc[0]
c = CI[(CI.event_key == q.event_key) & (CI.ham_key == q.ham_key)].iloc[0]
solC = np.asarray(qp.load_solution(c.sol_key)["sol"], float)
solQ = np.asarray(qp.load_solution(q.sol_key)["sol"], float); solQ /= np.linalg.norm(solQ)
ham = qp.build_hamiltonian(S._event_of(c), epsilon=float(c.epsilon), gamma=g, delta=d)
A = ham.A.toarray(); bvec = ham.b
w, Umat = np.linalg.eigh(A)
beta = Umat.T @ (bvec / np.linalg.norm(bvec))
tt = np.pi / A[0, 0]
xC_model = Umat @ (beta / w);            xC_model /= np.linalg.norm(xC_model)
xQ_model = np.abs(Umat @ (beta * np.cos(w*tt/2))); xQ_model /= np.linalg.norm(xQ_model)
solCn = solC / np.linalg.norm(solC)
print(f"n={n}: n_seg={A.shape[0]}, lambda in [{w.min():.3f},{w.max():.3f}], notch at lambda={A[0,0]:.0f}")
print(f"  false bulk on the notch: {(np.abs(w-A[0,0])<1e-6).sum()} eigenvalues == gamma+delta ({A[0,0]:.0f})")
print(f"  CLASSICAL  1/lambda model vs stored sol_C : max|diff|={np.max(np.abs(xC_model-solCn)):.2e}, corr={np.corrcoef(xC_model,solCn)[0,1]:.4f}")
print(f"  QUANTUM 1-bit cos-filter model vs sol_Q   : max|diff|={np.max(np.abs(xQ_model-solQ)):.3f}, corr={np.corrcoef(xQ_model,solQ)[0,1]:.4f}")
print("  (classical match is exact to machine precision; quantum residual = single-step Trotter of the non-commuting couplings)")

fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.8))
# (a) the two eigenfilters vs lambda, with the notch and the discrete spectrum
lg = np.linspace(1.5, 6.5, 400)
ax[0].plot(lg, 1/lg/np.max(1/lg), color="#1b7837", lw=2, label=r"classical $1/\lambda$ (norm.)")
ax[0].plot(lg, np.cos(lg*tt/2), color="#d6604d", lw=2, label=r"1BQF $\cos(\lambda t/2)$")
ax[0].axhline(0, color="k", lw=0.6); ax[0].axvline(diag, color="#2166ac", ls="--", lw=1.6, label=r"notch $\lambda=\gamma+\delta$")
chain_lam = diag - 2*np.cos(np.arange(1,5)*np.pi/5)
ax[0].scatter(chain_lam, np.cos(chain_lam*tt/2), color="#d6604d", zorder=5, s=45, ec="k")
ax[0].scatter([diag],[0], color="#2166ac", zorder=6, s=70, ec="k", label=r"false bulk ($\lambda=\gamma+\delta$)")
ax[0].set_xlabel(r"eigenvalue $\lambda$"); ax[0].set_ylabel("eigenmode weight")
ax[0].set_title("(a) Classical $1/\\lambda$ vs the 1-bit cosine filter", fontweight="bold")
ax[0].legend(fontsize=8)
# (b) real spectrum: false pile-up on the notch + true-chain satellites
ax[1].hist(w, bins=120, color="#6a3d9a", alpha=0.8)
ax[1].axvline(diag, color="#2166ac", ls="--", lw=1.6, label=r"notch $\lambda=\gamma+\delta=4$")
for cl in chain_lam:
    ax[1].axvline(cl, color="#d6604d", ls=":", lw=1)
ax[1].set_yscale("log"); ax[1].set_xlabel(r"eigenvalue $\lambda$ of stored $A$ (n=10)")
ax[1].set_ylabel("count"); ax[1].legend(fontsize=8)
ax[1].set_title("(b) Real spectrum: false bulk sits ON the notch", fontweight="bold")
# (c) model vs solver agreement on the real event
ax[2].scatter(solCn, xC_model, s=8, alpha=0.4, color="#1b7837", label=f"classical (corr {np.corrcoef(xC_model,solCn)[0,1]:.3f})")
ax[2].scatter(solQ, xQ_model, s=8, alpha=0.4, color="#d6604d", label=f"1BQF filter (corr {np.corrcoef(xQ_model,solQ)[0,1]:.3f})")
lim = max(solQ.max(), xQ_model.max(), solCn.max())*1.05
ax[2].plot([0,lim],[0,lim], "k--", lw=1)
ax[2].set_xlabel("stored solver activation"); ax[2].set_ylabel("eigenfilter-model activation")
ax[2].set_title("(c) The filters reproduce the stored solves", fontweight="bold"); ax[2].legend(fontsize=8)
fig.suptitle(r"First-principles proof: classical $=1/\lambda$ inversion, 1BQF $=\cos(\lambda t/2)$ 1-bit filter (notch at $\gamma+\delta$)",
             fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"eigenfilter_proof.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show(); print("saved eigenfilter_proof")

n=10: n_seg=400, lambda in [2.382,5.618], notch at lambda=4
  false bulk on the notch: 360 eigenvalues == gamma+delta (4)
  CLASSICAL  1/lambda model vs stored sol_C : max|diff|=1.92e-09, corr=1.0000
  QUANTUM 1-bit cos-filter model vs sol_Q   : max|diff|=0.039, corr=0.9815
  (classical match is exact to machine precision; quantum residual = single-step Trotter of the non-commuting couplings)


saved eigenfilter_proof


### 7.4 Classical vs quantum — the contrast, proven

| quantity | classical $A^{-1}\mathbf b$ | 1BQF (1-bit $\cos$ filter) |
|---|---|---|
| eigenmode weight | $1/\lambda$ (smooth, monotone, $>0$) | $\cos(\lambda t/2)$, $t=\pi/(\gamma+\delta)$ — **zero at $\lambda=\gamma+\delta$**, sign-changing |
| false / isolated ($\lambda=\gamma+\delta$) | $\delta/(\delta+\gamma)=0.25$ | **annihilated → 0** (sits on the notch) |
| true outer ($4/11$) | $0.3636$ | **$\approx0.18$ (halved)** |
| true inner ($5/11$) | $0.4545$ | $\approx0.45$ (preserved) |
| reproduces stored solve | to $10^{-9}$ | to corr $0.98$ ($\cos$ filter; rest = Trotter) |
| at $\tau=0.35$ | all true $>\tau>$ false → eff $\approx100\%$, far $\approx0$ | inner $>\tau>$ outer, false $\approx0$ → eff $\approx75\%$, far $\approx0$ |

**The one-line statement.** Both solvers see the *same* matrix with the *same*
Hopfield spectrum (false bulk at $\lambda=\gamma+\delta$, true chains at
$(\gamma+\delta)-2\cos\frac{k\pi}{5}$). The classical solver inverts it smoothly
($1/\lambda$) and so **preserves** the fixed-point levels that $\tau=0.35$ was
built for. The 1-bit quantum filter replaces $1/\lambda$ with
$\cos(\lambda t/2)$, whose **notch lands exactly on the false bulk** (annihilating
it) but whose non-monotone shape **down-shifts the outer true plateau by ~2×**.
The efficiency "loss" at $\tau=0.35$ is therefore *not* a solver failure and *not*
a metric bug — it is the fixed classical threshold cutting a true plateau that the
one-bit inversion has moved. (And the high-$T$ false-rate growth is the mirror
image: the few *coupled* false segments whose eigenvalues drift off the
$\lambda=\gamma+\delta$ notch escape annihilation — the false tail of §6.)

### 7.5 Emphasising the spectral mismatch — three views of the same fact

The classical inversion and the 1-bit quantum filter draw on **completely
different parts of the same spectrum**. Three complementary views (all on the
stored $n=10$ event unless noted):

* **(a) Re-weighting ratio** $R(\lambda)=\dfrac{\text{quantum weight}}{\text{classical weight}}=\dfrac{\cos(\lambda t/2)}{1/\lambda}=\lambda\cos(\lambda t/2)$. $R=1$ would mean "identical to classical"; instead $R$ is $0$ at the notch $\lambda=\gamma+\delta$, $>1$ for the lowest mode, and **negative (sign-flipped)** above the notch — reaching $-3.3$ at the top of the chain spectrum.
* **(b) Spectral activation mass** $\propto(\text{weight}_j\,\beta_j)^2$ per eigenvalue. The classical solution puts **$\approx77\%$ of its energy on the false bulk** at $\lambda=\gamma+\delta$; the 1-bit filter puts **exactly $0\%$ there** and $\approx99\%$ on the below-notch true modes. The two solvers literally live in different halves of the spectrum.
* **(c) It is structural in $\gamma$.** The chain spectrum $(\gamma+\delta)-2\cos\frac{k\pi}{5}$ is *centred on* $\gamma+\delta$, so the notch **always bisects it** (2 modes below, 2 above) for every $\gamma$ — the outer/inner plateau split is not special to $\gamma=3$.

In [12]:
# stored n=10 event, eigen-decomposition (reuse from §7b)
n = 10
q = QI[QI.n_trk == n].iloc[0]
c = CI[(CI.event_key == q.event_key) & (CI.ham_key == q.ham_key)].iloc[0]
ham = qp.build_hamiltonian(S._event_of(c), epsilon=float(c.epsilon), gamma=g, delta=d)
A = ham.A.toarray(); w, Umat = np.linalg.eigh(A)
beta = Umat.T @ (np.ones(A.shape[0]) / np.sqrt(A.shape[0]))
tt = np.pi / A[0, 0]; notch = A[0, 0]
mass_c = (beta / w) ** 2; mass_c /= mass_c.sum()
mass_q = (beta * np.cos(w*tt/2)) ** 2; mass_q /= mass_q.sum()
ev = np.round(w, 3); uev = np.unique(ev)
mc = np.array([mass_c[ev == e].sum() for e in uev])
mq = np.array([mass_q[ev == e].sum() for e in uev])

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
# (a) re-weighting ratio R(lambda) = lambda*cos(lambda t/2)
lg = np.linspace(1.8, 6.2, 500)
R = lg * np.cos(lg * tt / 2)
ax[0].axhspan(-4, 0, color="#d6604d", alpha=0.08)
ax[0].plot(lg, R, color="#762a83", lw=2.2)
ax[0].axhline(1, color="#1b7837", ls="--", lw=1.4, label="R = 1 (identical to classical)")
ax[0].axhline(0, color="k", lw=0.7)
ax[0].axvline(notch, color="#2166ac", ls="--", lw=1.6, label=r"notch $\lambda=\gamma+\delta$")
mult = np.array([(ev == e).sum() for e in uev])
ax[0].scatter(uev, uev*np.cos(uev*tt/2), s=20+6*mult, color="#762a83", ec="k", zorder=5)
ax[0].text(notch+0.05, -2.6, "sign-flipped\n(quantum reverses\nthese modes)", color="#d6604d", fontsize=8)
ax[0].set_xlabel(r"eigenvalue $\lambda$"); ax[0].set_ylabel(r"$R(\lambda)=\lambda\cos(\lambda t/2)$")
ax[0].set_title("(a) Quantum-vs-classical re-weighting per mode", fontweight="bold")
ax[0].legend(fontsize=8, loc="lower left")
# (b) spectral activation mass per eigenvalue
x = np.arange(len(uev)); ww = 0.4
ax[1].bar(x-ww/2, mc, ww, color="#1b7837", label="classical ($1/\\lambda$)")
ax[1].bar(x+ww/2, mq, ww, color="#d6604d", label="1BQF ($\\cos\\lambda t/2$)")
ax[1].set_xticks(x); ax[1].set_xticklabels([f"{e:.2f}" for e in uev])
inotch = int(np.argmin(np.abs(uev-notch)))
ax[1].annotate("false bulk\nclassical 77% / quantum 0%", xy=(x[inotch], max(mc[inotch], 0.02)),
               xytext=(x[inotch]-0.2, 0.55), fontsize=8, color="#2166ac",
               arrowprops=dict(arrowstyle="->", color="#2166ac"))
ax[1].set_xlabel(r"eigenvalue $\lambda$ (notch at $\gamma+\delta=%g$)" % notch)
ax[1].set_ylabel("fraction of solution energy")
ax[1].set_title("(b) Where each solver's solution lives in the spectrum", fontweight="bold")
ax[1].legend(fontsize=9)
# (c) gamma-structural: notch always bisects the chain spectrum
for gg in (1.0, 2.0, 3.0):
    chain = (gg + d) - 2*np.cos(np.arange(1, 5)*np.pi/5)
    ax[2].scatter(chain, [gg]*4, s=70, color="#1b7837", ec="k", zorder=4,
                  label="chain eigenvalues" if gg == 1.0 else None)
    ax[2].scatter([gg+d], [gg], marker="v", s=130, color="#2166ac", ec="k", zorder=5,
                  label=r"notch $\lambda=\gamma+\delta$" if gg == 1.0 else None)
    ax[2].plot([chain.min()-0.3, chain.max()+0.3], [gg, gg], color="grey", lw=0.8, zorder=1)
ax[2].set_yticks([1, 2, 3]); ax[2].set_ylabel(r"$\gamma$ ($\delta=1$)")
ax[2].set_xlabel(r"eigenvalue $\lambda$")
ax[2].set_title("(c) The notch bisects the chain spectrum for every $\\gamma$", fontweight="bold")
ax[2].legend(fontsize=8, loc="upper left")
fig.suptitle("Spectral mismatch: classical inverts smoothly across the spectrum; the 1-bit filter "
             "wipes the notch and reverses the high modes", fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"spectral_mismatch.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show()
print(f"saved spectral_mismatch | classical mass on notch = {mc[inotch]:.3f}, quantum = {mq[inotch]:.3f}")
print(f"  below-notch mass: classical {mass_c[w<notch].sum():.3f}  quantum {mass_q[w<notch].sum():.3f}")

saved spectral_mismatch | classical mass on notch = 0.769, quantum = 0.000
  below-notch mass: classical 0.228  quantum 0.991


In [13]:
# A second emphasis: the filter f(lambda)=cos(lambda t/2) ACROSS gamma, each over its own
# chain spectrum — same notch-bisection, same outer-suppression, rescaled (self-similar).
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
cols = {1.0: "#4575b4", 2.0: "#984ea3", 3.0: "#d73027"}
for gg in (1.0, 2.0, 3.0):
    dg = gg + d; tg = np.pi / dg
    lg = np.linspace(0.2, 2*dg-0.2, 400)
    ax[0].plot(lg, np.cos(lg*tg/2), color=cols[gg], lw=2, label=fr"$\gamma={gg:g}$ (notch {dg:g})")
    chain = dg - 2*np.cos(np.arange(1, 5)*np.pi/5)
    ax[0].scatter(chain, np.cos(chain*tg/2), color=cols[gg], s=45, ec="k", zorder=5)
    ax[0].axvline(dg, color=cols[gg], ls=":", lw=1)
ax[0].axhline(0, color="k", lw=0.7)
ax[0].set_xlabel(r"eigenvalue $\lambda$"); ax[0].set_ylabel(r"1-bit filter $\cos(\lambda t/2)$")
ax[0].set_title("(a) The filter and its notch for $\\gamma\\in\\{1,2,3\\}$", fontweight="bold")
ax[0].legend(fontsize=8)
# normalised by the notch: collapse to a single universal curve
xu = np.linspace(0.05, 1.95, 400)
ax[1].plot(xu, np.cos(xu*np.pi/2), color="k", lw=2.3, label=r"universal $\cos(\frac{\pi}{2}\frac{\lambda}{\gamma+\delta})$")
for gg in (1.0, 2.0, 3.0):
    dg = gg + d
    chain = (dg - 2*np.cos(np.arange(1, 5)*np.pi/5)) / dg
    ax[1].scatter(chain, np.cos(chain*np.pi/2), color=cols[gg], s=55, ec="k", zorder=5,
                  label=fr"$\gamma={gg:g}$ chain modes")
ax[1].axvline(1.0, color="#2166ac", ls="--", lw=1.5, label="notch (λ/(γ+δ)=1)")
ax[1].axhline(0, color="k", lw=0.7)
ax[1].set_xlabel(r"$\lambda/(\gamma+\delta)$"); ax[1].set_ylabel(r"filter value")
ax[1].set_title("(b) Rescaled by the notch: one universal filter, modes always straddle it",
                fontweight="bold")
ax[1].legend(fontsize=8)
fig.suptitle(r"Self-similarity in $\gamma$: the chain spectrum is centred on $\gamma+\delta$, so the 1-bit notch always splits true segments into two plateaus",
             fontsize=11.5, fontweight="bold", y=1.02)
fig.tight_layout()
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"notch_universal_gamma.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show(); print("saved notch_universal_gamma")

saved notch_universal_gamma


## 8. Verdict

**Convention error — not definitional, not mismatched metrics.**

| Candidate | Finding |
|---|---|
| Mismatched metrics | **Ruled out.** Q and C are paired on the same `(event_key, ham_key)` → same matrix A, identical truth mask, equal-length vectors (§1). |
| Definitional error | **Ruled out.** Feeding `sol_C` through the quantum metric path reproduces the classical false rate exactly (§2). |
| Convention error | **Confirmed.** The quantum metric rescaled `sol_Q` to ‖sol_C‖, which is 95–99 % the n-dependent 0.25 false bulk (§3), so the rescale factor ∝ √n_seg inflated amplitudes and pushed borderline false segments over the fixed τ = 0.35 as n grows (§4). Rescaling on the classical signal support collapses the quantum false rate onto the classical curve, and AUC ≈ 1 shows the quantum ranking is essentially perfect (§5). |

**Why it happened:** the absolute threshold τ = δ/(δ+γ)+0.10 is calibrated to the
classical Hopfield **fixed points**, which are n-independent. The 1BQF statevector
solution is unit-norm; rescaling it to ‖sol_C‖ (a false-bulk-dominated, growing
quantity) destroyed that calibration.

**Shipped fix:** `qtrk_pipeline/metrics.py::rescale_to_signal`, called by
`quantum_metrics` — rescale `sol_Q` to ‖sol_C‖ **restricted to the classical
active support** (`sol_C > τ`), the false-bulk-free signal scale (truth-free,
no re-solving), and report `cos_QC` on that same support. Side effect worth
noting: the corrected quantum **efficiency** at γ=3 settles at ~75 % (the old
~99.5 % was the *same* artefact inflating true amplitudes too) — i.e. the 1BQF
genuinely under-reconstructs ~25 % of true segments, previously masked. The
underlying quantum solve is faithful where it counts (cos on signal ≈ 0.97,
AUC ≈ 1).